# Evaluating the turnover model

Evaluates a turnover model bundle against the project's **fixed, seed-locked holdout cohort** — the exact benchmark `companysim.ml.gate.run_training_gate` uses to decide whether a candidate is safe to promote. The holdout's `DatasetConfig`, seed, and horizon are imported directly from `ml.gate` so this notebook can never silently drift from the real promotion benchmark.

Loads the current production bundle (`models/turnover_production.joblib`) if one exists; otherwise trains a small one on the spot so this notebook runs standalone on a fresh clone (that model file is git-ignored, so a fresh checkout won't have one yet).

In [1]:
import os
from pathlib import Path

while not (Path.cwd() / 'pyproject.toml').exists():
    os.chdir('..')
print(f'repo root: {Path.cwd()}')

repo root: D:\companysim


In [2]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.calibration import calibration_curve
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)

from companysim.ml.diagnostics import _aggregate_feature_importances
from companysim.ml.gate import (
    EVAL_CONFIG,
    EVAL_HORIZON_TICKS,
    EVAL_REPLICATES,
    EVAL_SIM_SEED,
    PRODUCTION_PATH,
    run_training_gate,
)
from companysim.ml.registry import TurnoverModelBundle, load_bundle
from companysim.ml.train import precision_at_k
from companysim.ml.turnover_features import CATEGORICAL_FEATURES, FEATURE_COLUMNS, build_feature_frame
from companysim.ml.turnover_labels import build_turnover_cohort

pd.set_option('display.max_columns', None)

## 1. Load a bundle to evaluate


In [3]:
try:
    bundle = load_bundle(PRODUCTION_PATH, expected_type=TurnoverModelBundle)
    print(f'Loaded production bundle -> {PRODUCTION_PATH.resolve()}')
except FileNotFoundError:
    print('No production bundle yet on this machine -- bootstrapping a small one for this notebook.')
    result = run_training_gate(headcount=800, replicates=2, horizon=8, force_promote=True)
    bundle = load_bundle(PRODUCTION_PATH, expected_type=TurnoverModelBundle)
    print(f'Bootstrapped and saved -> {PRODUCTION_PATH.resolve()}')

bundle.metadata

Loaded production bundle -> D:\companysim\models\turnover_production.joblib


{'seed': 2024,
 'auc': 0.6651844155844155,
 'precision_at_10': 0.16981132075471697,
 'precision_at_20': 0.15283018867924528,
 'base_rate': 0.0659433962264151,
 'n_train': 7950,
 'n_test': 2650,
 'trained_at': '2026-07-07T12:30:00.199059+00:00',
 'training_seed': 2024,
 'training_headcount': 2000,
 'holdout_eval': {'auc': 0.6371305837190506,
  'precision_at_10': 0.10222222222222223,
  'precision_at_20': 0.12222222222222222,
  'base_rate': 0.06933333333333333,
  'n': 4500},
 'n_live_examples': 2600}

## 2. Build the fixed holdout cohort

Same `DatasetConfig`/seed/horizon `ml.gate.evaluate_bundle_on_holdout` uses — never regenerated with a different seed, so it's a stable benchmark across every retrain, and never mixed into training data.

In [4]:
holdout = build_turnover_cohort(
    EVAL_CONFIG, horizon_ticks=EVAL_HORIZON_TICKS,
    replicates=EVAL_REPLICATES, sim_base_seed=EVAL_SIM_SEED,
)
holdout_feats = build_feature_frame(holdout.tables)
holdout_merged = holdout.labels.merge(holdout_feats, on='employee_id')

y_true = holdout_merged['quit_within_horizon'].astype(int).to_numpy()
proba = bundle.classifier.predict_proba(holdout_merged[list(FEATURE_COLUMNS)])[:, 1]

print(f'Holdout: {len(y_true):,} rows, base rate {y_true.mean():.2%}')

Holdout: 4,500 rows, base rate 6.93%


## 3. Headline metrics


In [5]:
metrics = {
    'auc': roc_auc_score(y_true, proba) if len(set(y_true)) > 1 else float('nan'),
    'average_precision': average_precision_score(y_true, proba),
    'precision_at_10': precision_at_k(y_true, proba, 0.10),
    'precision_at_20': precision_at_k(y_true, proba, 0.20),
    'precision_at_30': precision_at_k(y_true, proba, 0.30),
    'base_rate': float(y_true.mean()),
    'n': len(y_true),
}
pd.DataFrame([metrics]).T.rename(columns={0: 'value'})

,value
auc,0.637131
average_precision,0.104423
precision_at_10,0.102222
precision_at_20,0.122222
precision_at_30,0.106667
base_rate,0.069333
n,4500.000000


## 4. ROC curve

Realistic band is roughly 0.55-0.85 for this synthetic task — see `tests/test_turnover_pipeline.py::test_turnover_model_auc_in_realistic_band` for why both ends of that band matter: too low means the pipeline lost signal, too high (near 1.0) would suggest leakage.

In [6]:
fpr, tpr, _ = roc_curve(y_true, proba)
fig = go.Figure()
fig.add_trace(go.Scatter(x=fpr, y=tpr, mode='lines', name=f"ROC (AUC={metrics['auc']:.3f})", line=dict(color='#4f46e5', width=3)))
fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode='lines', name='Chance', line=dict(color='gray', dash='dash')))
fig.update_layout(
    title='ROC curve', xaxis_title='False positive rate', yaxis_title='True positive rate',
    template='plotly_white', width=650, height=500,
)
fig.show()

## 5. Precision-recall curve


In [7]:
prec, rec, _ = precision_recall_curve(y_true, proba)
fig = go.Figure()
fig.add_trace(go.Scatter(x=rec, y=prec, mode='lines', name=f"PR (AP={metrics['average_precision']:.3f})", line=dict(color='#4f46e5', width=3)))
fig.add_hline(y=metrics['base_rate'], line_dash='dash', line_color='gray', annotation_text='base rate')
fig.update_layout(
    title='Precision-recall curve', xaxis_title='Recall', yaxis_title='Precision',
    template='plotly_white', width=650, height=500,
)
fig.show()

## 6. Calibration (reliability diagram)

Do predicted probabilities mean what they say? A well-calibrated model's points sit near the diagonal — e.g. among employees scored ~30% risk, about 30% should actually quit.

In [8]:
frac_pos, mean_pred = calibration_curve(y_true, proba, n_bins=10, strategy='quantile')
fig = go.Figure()
fig.add_trace(go.Scatter(x=mean_pred, y=frac_pos, mode='lines+markers', name='Model', line=dict(color='#4f46e5', width=3)))
fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode='lines', name='Perfectly calibrated', line=dict(color='gray', dash='dash')))
fig.update_layout(
    title='Calibration curve', xaxis_title='Mean predicted probability', yaxis_title='Observed frequency',
    template='plotly_white', width=650, height=500,
)
fig.show()

## 7. Confusion matrix + classification report

At the model's own `high_risk_threshold` (used for the At-Risk page's risk tiers). Note: with only ~7% of employees quitting in the holdout, a naive 0.5 threshold would predict almost nobody as "Quit" and still score >90% accuracy while catching none of them. `high_risk_threshold` is tuned lower than 0.5 for exactly this reason — recall on the minority class matters more than raw accuracy here. This is also why the project leans on precision@k (below) rather than a single fixed threshold for the At-Risk page.

In [9]:
threshold = bundle.high_risk_threshold
y_pred = (proba >= threshold).astype(int)
cm = confusion_matrix(y_true, y_pred)

fig = px.imshow(
    cm, text_auto=True, color_continuous_scale='Purples',
    labels=dict(x='Predicted', y='Actual', color='Count'),
    x=['Stayed', 'Quit'], y=['Stayed', 'Quit'],
    title=f'Confusion matrix @ threshold={threshold:.2f}',
)
fig.update_layout(template='plotly_white', width=500, height=450)
fig.show()

print(classification_report(y_true, y_pred, target_names=['Stayed', 'Quit']))

              precision    recall  f1-score   support

      Stayed       0.93      1.00      0.96      4188
        Quit       0.33      0.00      0.01       312

    accuracy                           0.93      4500
   macro avg       0.63      0.50      0.49      4500
weighted avg       0.89      0.93      0.90      4500



## 8. Score separation by true label


In [10]:
sep_df = pd.DataFrame({'probability': proba, 'label': np.where(y_true == 1, 'Quit', 'Stayed')})
fig = px.histogram(
    sep_df, x='probability', color='label', barmode='overlay', nbins=40,
    color_discrete_map={'Stayed': '#4f46e5', 'Quit': '#dc2626'}, opacity=0.65,
    title='Predicted probability distribution by true outcome',
)
fig.update_layout(template='plotly_white')
fig.show()

## 9. Precision@k

The metric a retention program actually cares about: *if we can only intervene on the riskiest k% of the workforce, how many of them were really going to quit?* (see `ml/train.py::precision_at_k`).

In [11]:
ks = [0.05, 0.10, 0.20, 0.30, 0.50]
prec_at_k = [precision_at_k(y_true, proba, k) for k in ks]
fig = px.bar(
    x=[f'{int(k*100)}%' for k in ks], y=prec_at_k,
    labels={'x': 'Top-k% scored highest risk', 'y': 'Precision'},
    title='Precision@k vs. base rate', template='plotly_white',
)
fig.update_traces(marker_color='#4f46e5')
fig.add_hline(y=metrics['base_rate'], line_dash='dash', line_color='gray', annotation_text='base rate')
fig.show()

## 10. Feature importance


In [12]:
importances = _aggregate_feature_importances(bundle.classifier, CATEGORICAL_FEATURES)
imp_df = (
    pd.Series(importances, name='importance')
    .sort_values(ascending=True)
    .reset_index()
    .rename(columns={'index': 'feature'})
)
fig = px.bar(
    imp_df, x='importance', y='feature', orientation='h',
    title='Feature importance (production/evaluated bundle)', template='plotly_white',
)
fig.update_traces(marker_color='#4f46e5')
fig.show()

## Notes

- Methodology and threshold rationale: `docs/diagnosis_thresholds.md`, `docs/project_overview.md` (§6 The ML layer).
- This same holdout + AUC-regression check is what gates every promotion in `ml.gate.run_training_gate`, including retrains that blend in live examples collected from real webapp usage (§11 MLOps in `docs/project_overview.md`).